In [ ]:
from torch import float32, tensor, randn, zeros
from torch.autograd import Function
from torch.nn import Module, Parameter

import import_ipynb
from common import assert_eq, T # type: ignore

$$ f=x+1 $$
$$ \frac{\partial f}{\partial x} = 1 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=3+f $$
$$ \frac{\partial F}{\partial f} = 1 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} = 1$$

In [ ]:
class Add1Function(Function):
    @staticmethod
    def forward(ctx, x):
        return x + 1

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 1.0)

        df_dx = 1
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   

class Add1(Module):
    def forward(self, x):
        return Add1Function.apply(x)
    
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)
   

def _test_Add1():
    x = tensor(4, dtype=float32, requires_grad=True)
    F = 3 + Add1()(x)     
    F.backward()     

    assert_eq(F.item(), 3.0 + (4.0 + 1))
    assert_eq(x.grad.item(), 1.0)


if __name__ == "__main__":
    _test_Add1()

$$ f=2x $$
$$ \frac{\partial f}{\partial x} = 2 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=1+3f $$
$$ \frac{\partial F}{\partial f} = 3 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$

In [ ]:
class Mul2Function(Function):
    @staticmethod
    def forward(ctx, x):
        return 2 * x

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 3.0)
        
        df_dx = 2
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   

class Mul2(Module):
    def forward(self, x):
        return Mul2Function.apply(x)
   
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)


def _test_Mul2():
    x = tensor([4.0], requires_grad=True)
    F = 1 + 3 * Mul2()(x)     
    F.backward()     

    assert_eq(F.item(), 1.0 + 3.0 * (2.0 * 4.0))
    assert_eq(x.grad.item(), 3.0 * 2.0)


if __name__ == "__main__":
    _test_Mul2()

$$ f=xy $$
$$ \frac{\partial f}{\partial x} = y $$
$$ \frac{\partial f}{\partial y} = x $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=2+5f $$
$$ \frac{\partial F}{\partial f} = 5 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$
$$ \frac{\partial F}{\partial y} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial y} $$

In [ ]:
class MultiplyFunction(Function):
    @staticmethod
    def forward(ctx, x, y):
        # `save_for_backward` is accessible thanks to the `Function` base class.
        ctx.save_for_backward(x, y)

        return x * y

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 5.0)

        # `saved_tensors` is accessible thanks to the `Function` base class.
        (x, y) = ctx.saved_tensors

        df_dx = y
        dF_dx =  dF_df * df_dx

        df_dy = x
        dF_dy =  dF_df * df_dy

        return (dF_dx, dF_dy)
   

# `MultiplyFunction` implements the actual operator.
# `Multiply(Module)` is just a wrapper for nicer API.
class Multiply(Module):
    def forward(self, x, y):
        return MultiplyFunction.apply(x, y)
   
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)


def _test_MultiplyFunction():
    x = tensor([3.0], requires_grad=True)
    y = tensor([4.0], requires_grad=True)

    F = 2.0 + 5.0 * Multiply()(x, y)     
    F.backward()        

    assert_eq(F.item(), 2.0 + 5.0 * (3.0 * 4.0))
    assert_eq(x.grad.item(), 4.0 * 5.0)
    assert_eq(y.grad.item(), 3.0 * 5.0)


if __name__ == "__main__":
    _test_MultiplyFunction()